# Ray Train Architecture
© 2025, Anyscale. All Rights Reserved

This notebook will walk you through Ray Train architecture and integrating Ray Train with Ray Data.

<div class="alert alert-block alert-info">

<b> Here is the roadmap for this notebook </b>

<ol>
  <li>Ray Train Architecture</li>
  <li>When to Integrate Ray Train with Ray Data</li>
  <li>Integration Architecture</li>
  <li>Integrating Ray Train with Ray Data</li>
  <li>Training with Ray Train and Ray Data</li>
  <li>Using a Heterogeneous Cluster</li>
  <li>Last-Mile Data Preprocessing</li>
</ol>
</div>

**Imports**

In [ ]:
import os   
import tempfile

from PIL import Image
import numpy as np
import pandas as pd
import torch
from torch.nn import CrossEntropyLoss
from torch.optim import Adam
from torchvision.models import resnet18
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Normalize, Compose

import ray
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer

**Utilities**

Below are utility functions for building the model, logging metrics, and saving checkpoints.

In [ ]:
def build_resnet18():
    model = resnet18(num_classes=10)
    model.conv1 = torch.nn.Conv2d(
        in_channels=1,  # grayscale MNIST images
        out_channels=64,
        kernel_size=(7, 7),
        stride=(2, 2),
        padding=(3, 3),
        bias=False,
    )
    return model


def load_model_ray_train() -> torch.nn.Module:
    model = build_resnet18()
    model = ray.train.torch.prepare_model(model)  # Instead of model = model.to("cuda")
    return model


def print_metrics_ray_train(loss: torch.Tensor, epoch: int) -> None:
    metrics = {"loss": loss.item(), "epoch": epoch}
    world_rank = ray.train.get_context().get_world_rank()  # report from all workers
    print(f"{metrics=} {world_rank=}")
    return metrics


def save_checkpoint_and_metrics_ray_train(
    model: torch.nn.Module, metrics: dict[str, float]
) -> None:
    with tempfile.TemporaryDirectory() as temp_checkpoint_dir:
        torch.save(
            model.module.state_dict(),  # note the `.module` to unwrap the DistributedDataParallel
            os.path.join(temp_checkpoint_dir, "model.pt"),
        )

        ray.train.report(
            metrics,
            checkpoint=ray.train.Checkpoint.from_directory(temp_checkpoint_dir),
        )

## 1. Ray Train Architecture

Ray Train's architecture is based on the following components:
1. A Ray Train Controller/Driver that schedules the training workers, handles errors and manages checkpoints
2. Ray Train Worker executing the training code

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/train_architecture.jpg" loading="lazy" width="700">

### Key API concepts

Below are the key API concepts of Ray Train:

1. `train_loop_per_worker`: This is the main function that contains your model training logic.
2. `ScalingConfig`: This is used to specify the number of workers and compute resources (CPUs or GPUs, TPUs).
3. `Trainer`: This is used to manage the training process.
4. `Trainer.fit()`: This is used to start the training process.

<img src="https://docs.ray.io/en/latest/_images/overview.png" width="700" loading="lazy">

### Key components

Below is the diagram showing how the key components usually work together.

- The Train Controller/Driver is constantly performing health checks on the Train workers.
- The Train workers are running the training loop and at a particular frequency checkpointing the model to a persistent storage.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/ray_train_detailed_architecture.png" width="800" loading="lazy">

## 2. How does Ray Data differ from using a framework-specific DataLoader

Let's compare how using Ray Data differs from using a framework-specific DataLoader (e.g PyTorch DataLoader or TensorFlow Dataset)

### Compare differences on a single node setup

Let's first look at how PyTorch DataLoader vs Ray Data work in a single node setup.


Below is a diagram showing how Pytorch DataLoader works in a single node setup.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/pytorch_dataloader_single_worker.png" width="800" loading="lazy">    

Below is a diagram showing how Ray Data works in a single node setup.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/ray_data_ray_train_single_worker.png" width="800" loading="lazy">    

### Compare differences on a distributed setup

Let's now look at how PyTorch DataLoader vs Ray Data work in a distributed setup.


Below is a diagram showing how Pytorch DataLoader works in a distributed setup.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/pytorch_dataloader_multi_worker.png" width="800" loading="lazy">

Below is a diagram showing how Ray Data works in a distributed setup.

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/ray_data_ray_train_multi_worker.png" width="1000" loading="lazy">


## 2. When to Integrate Ray Train with Ray Data
Use both Ray Train and Ray Data when you face one of the following challenges:
| Challenge | Detail | Solution |
| --- | --- | --- |
| Need a **consistent interface for loading and sharding** data | The training process may need to load data from various sources, such as Parquet, CSV, or lakehouses. | Ray Data provides a standardize approach for loading, shuffling, sharding, and streaming batch data to your distributed training run. |
| Need to perform online or **just-in-time data processing** on **CPU and GPU nodes** | You are in a scenario where it makes more sense to perform data preprocessing on the fly instead of preprocess your data in batch in a separate job. This usually is the case if preprocessing your data takes an extremely long time (on the order of weeks) and you don't want this to block your training run. | Ray Train's integration with Ray Data makes it easy to implement just-in-time data processing. |

## 3. Integration Architecture

Here is a diagram showing the a sample Ray Data and Ray Train integration for a simple homogenous cluster setup

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/homogeneous_training_ray_train_ray_data.png" width="900" loading="lazy">

## 4. Integrating Ray Train with Ray Data

Here is how our training loop will look like using **Ray Data** instead of the **PyTorch DataLoader**:

In [ ]:
def train_loop_ray_train_ray_data(config: dict):
    # Same initialization as before
    criterion = CrossEntropyLoss()
    model = load_model_ray_train()
    optimizer = Adam(model.parameters(), lr=1e-3)
    
    # This time we use Ray Train's integration with Ray Data to load the data
    global_batch_size = config["global_batch_size"]
    batch_size = global_batch_size // ray.train.get_context().get_world_size()
    data_loader = build_data_loader_with_ray_data(batch_size=batch_size) 
    

    for epoch in range(config["num_epochs"]):
        # No longer need to ensure data is on the correct device
        # data_loader.sampler.set_epoch(epoch)

        # Note our batches are now dictionaries instead of tuples
        for batch in data_loader: 
            outputs = model(batch["image"])
            loss = criterion(outputs, batch["label"])
            optimizer.zero_grad()
            loss.backward() 
            optimizer.step()


        metrics = print_metrics_ray_train(loss, epoch)
        save_checkpoint_and_metrics_ray_train(model, metrics)


Here is the updated `build_data_loader_with_ray_data` function that uses Ray Data to load the data:

In [ ]:
def build_data_loader_with_ray_data(
    batch_size: int, prefetch_batches: int = 2
):
    dataset_iterator = ray.train.get_dataset_shard("train")
    data_loader = dataset_iterator.iter_torch_batches(
        batch_size=batch_size, prefetch_batches=prefetch_batches
    )
    return data_loader

<div class="alert alert-block alert-info">

**Note** Use the `iter_torch_batches` function to build a torch compatible data loader.

</div>

### 4.1 Preparing the data
Let's store the training data in a format that Ray Data can easily read. 

Let's use the Parquet format, which is a columnar storage format that is efficient for reading and writing data.

In [ ]:
torch_dataset = MNIST(root="/mnt/local_storage/data", train=True, download=True)
df = pd.DataFrame({"image": torch_dataset.data.tolist(), "label": torch_dataset.targets})

def transform_images(row: dict):
    # Define the torchvision transform.
    transform = Compose([ToTensor(), Normalize((0.5,), (0.5,))])
    image_arr = np.array(row["image"], dtype=np.uint8)
    row["image"] = transform(Image.fromarray(image_arr))
    return row

# Use this approach if the transformations are extremely random
# and generating the transformed dataset takes a very long time (order of days)
ds = ray.data.from_pandas(df)
ds.write_parquet("/mnt/cluster_storage/raw_data")

# Use this approach if the transformed data can be reused across training runs and
# generating the transformed dataset takes a reasonable time (order of hours)
ds.map(transform_images).write_parquet("/mnt/cluster_storage/transformed_data")

### 4.2 Reading transformed data into a trainer

Next, construct a Ray Data Dataset from the Parquet source.

In [ ]:
# note we use resource tags to ensure Read Tasks run on the same node as the training worker
train_ds = ray.data.read_parquet(
    "/mnt/cluster_storage/transformed_data",
    ray_remote_args={"resources": {"accelerator_type:T4": 0.0001}},
)

## 5. Training with Ray Train and Ray Data

Let's define the scaling config

In [ ]:
# Make sure we use the same T4 GPUs for training
scaling_config = ScalingConfig(
    num_workers=2,
    use_gpu=True,
    resources_per_worker={"accelerator_type:T4": 0.0001},
)

Pass the constructed `train_ds` to the `TorchTrainer` via the `datasets` parameter.

In [ ]:
# By default datasets will be split equally
datasets = {"train": train_ds}

trainer = TorchTrainer(
    train_loop_ray_train_ray_data,
    train_loop_config={"num_epochs": 1, "global_batch_size": 512},
    scaling_config=scaling_config,
    run_config=RunConfig(
            storage_path="/mnt/cluster_storage/training/",
            name="dist-mnist-res18-ray-data-homogeneous",
    ),
    datasets=datasets,
)

Calling `trainer.fit()` will now use Ray Data to load and shard the data.

In [ ]:
trainer.fit()

## 6. Using a Heterogeneous Cluster

In case your training is **bottlenecked by data ingest**, then it might make sense to add dedicated "Data worker nodes" to increase the throughput. 

This usually happens in a regime of reading many files over a **limited network bandwidth** onto a **relatively small model** whose forward and backward pass finish relatively quickly. 

Here is a diagram showing the a sample Ray Data and Ray Train integration for a simple heterogeneous cluster setup

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/heterogeneous_training_ray_train_ray_data.png" width="900" loading="lazy">

To achieve a heterogenous run, add CPU worker nodes to the cluster and avoid constraining the read

In [ ]:
train_ds = ray.data.read_parquet("/mnt/cluster_storage/transformed_data")

What follows is the same approach to get the training job running

In [ ]:
datasets = {"train": train_ds}
trainer = TorchTrainer(
    train_loop_ray_train_ray_data,
    train_loop_config={"num_epochs": 1, "global_batch_size": 512},
    scaling_config=scaling_config,
    run_config=RunConfig(
        storage_path="/mnt/cluster_storage/training/",
        name="hetero-dist-mnist-res18-ray-data",
    ),
    datasets=datasets,
)

trainer.fit()

## 7. Last-Mile Data Preprocessing

In case producing your transformed dataset takes a very long time (days to weeks), and the transformations are quite random in nature (e.g. random cropping/sampling). It might make more sense to perform on-the-fly (last-mile data-preprocessing) into your training ingest.


Here is the architecture diagram we showed above in a heterogeneous setup

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-train-deep-dive/ray_train_v2_architecture.png">

To achieve last-mile preprocessing specify the transformations to be applied after the read

In [ ]:
train_ds = ray.data.read_parquet("/mnt/cluster_storage/raw_data/").map(transform_images)

What follows is the same approach to get the training job running

In [ ]:
datasets = {"train": train_ds}
trainer = TorchTrainer(
    train_loop_ray_train_ray_data,
    train_loop_config={"num_epochs": 1, "global_batch_size": 512},
    scaling_config=scaling_config,
    run_config=RunConfig(
        storage_path="/mnt/cluster_storage/training/",
        name="last-mile-dist-mnist-res18-ray-data",
    ),
    datasets=datasets,
)

trainer.fit()